# Work with Wind Station Service

This notebook is used to work with the wind station service.

In [1]:
%cd ../
%load_ext autoreload
%autoreload 2

/Users/matthaei/Documents/code/python/bachelor-project


In [18]:
MIGRATE_DATABASE = False

In [3]:
from src.weather_stations.weather_station_service import WeatherStationService
from src.database.database_service import DatabaseService
from omegaconf import DictConfig, OmegaConf
from hydra import compose, initialize_config_dir
import os

In [4]:
# Initialize Hydra configuration
config_dir = os.path.abspath("./conf")

# Initialize Hydra with the config directory
with initialize_config_dir(config_dir=config_dir, version_base=None):
    cfg = compose(config_name="config")

In [5]:
database_service = DatabaseService(cfg)

if MIGRATE_DATABASE:
    database_service.create_tables()


2025-09-02 17:07:04,953 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2025-09-02 17:07:04,954 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-09-02 17:07:05,008 INFO sqlalchemy.engine.Engine select current_schema()
2025-09-02 17:07:05,009 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-09-02 17:07:05,051 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2025-09-02 17:07:05,051 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-09-02 17:07:05,166 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-09-02 17:07:05,168 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s::VARCHAR AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s::VARCHAR, %(param_2)s::VARCHAR, %(param_3)s::VARCHAR, %(param_4)s::VARCHAR, %(param_5)s::VARCHAR]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class

2025-09-02 17:07:05.237 | INFO     | src.database.database_service:create_tables:14 - Tables created


## Fill database with weather stations from DWD

In [ ]:
weather_station_service = WeatherStationService(cfg, database_service)

In [ ]:
df = weather_station_service.fill_database_with_weather_stations()

2025-09-02 17:13:42.442 | INFO     | src.weather_stations.weather_station_data_provider:download_weather_stations_file:25 - Downloaded weather stations file
2025-09-02 17:13:42.449 | INFO     | src.weather_stations.weather_station_data_provider:parse_weather_stations_file:131 - Parsed 1378 stations
2025-09-02 17:13:42.453 | INFO     | src.weather_stations.weather_station_data_provider:process_weather_stations_df:13 - Processed 1378 stations


2025-09-02 17:13:42,498 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-09-02 17:13:42,499 INFO sqlalchemy.engine.Engine SELECT weather_stations.weather_station_id, weather_stations.name, weather_stations.latitude, weather_stations.longitude, weather_stations.height, weather_stations.state, weather_stations.start_date, weather_stations.end_date, weather_stations.is_active 
FROM weather_stations 
WHERE weather_stations.weather_station_id = %(weather_station_id_1)s::INTEGER
2025-09-02 17:13:42,499 INFO sqlalchemy.engine.Engine [cached since 376.6s ago] {'weather_station_id_1': 1}
2025-09-02 17:13:42,550 INFO sqlalchemy.engine.Engine SELECT weather_stations.weather_station_id, weather_stations.name, weather_stations.latitude, weather_stations.longitude, weather_stations.height, weather_stations.state, weather_stations.start_date, weather_stations.end_date, weather_stations.is_active 
FROM weather_stations 
WHERE weather_stations.weather_station_id = %(weather_station_id_1)s::INTEGER
2

In [17]:
df.head()

,weather_station_id,start_date,end_date,height,latitude,longitude,name,state,accessible,is_active
0,1,1937-01-01,1986-06-30,478,47.8413,8.8493,Aach,Baden-Württemberg,True,False
1,3,1891-01-01,2011-03-31,202,50.7827,6.0941,Aachen,Nordrhein-Westfalen,True,False
2,11,1980-09-01,2025-09-01,680,47.9736,8.5205,Donaueschingen (Landeplatz),Baden-Württemberg,True,True
3,44,1969-01-01,2025-09-01,44,52.9336,8.2370,Großenkneten,Niedersachsen,True,True
4,52,1969-01-01,2001-12-31,46,53.6623,10.1990,Ahrensburg-Wulfsdorf,Schleswig-Holstein,True,False


## Load from database

In [19]:
new_weather_station_service = WeatherStationService(cfg, database_service)

In [24]:
new_df = new_weather_station_service.load_from_database()
new_df.head()

2025-09-02 17:42:19,518 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-09-02 17:42:19,519 INFO sqlalchemy.engine.Engine SELECT weather_stations.weather_station_id, weather_stations.name, weather_stations.latitude, weather_stations.longitude, weather_stations.height, weather_stations.state, weather_stations.start_date, weather_stations.end_date, weather_stations.is_active 
FROM weather_stations 
WHERE weather_stations.is_active = true AND weather_stations.state = %(state_1)s::VARCHAR ORDER BY weather_stations.weather_station_id
2025-09-02 17:42:19,519 INFO sqlalchemy.engine.Engine [generated in 0.00036s] {'state_1': 'Brandenburg'}


2025-09-02 17:42:19.565 | INFO     | src.weather_stations.weather_station_data_provider:load_from_database:15 - Loaded 26 weather stations from database


2025-09-02 17:42:19,566 INFO sqlalchemy.engine.Engine ROLLBACK


,height,longitude,start_date,is_active,name,weather_station_id,latitude,state,end_date
0,50.0,12.8518,2019-04-09,True,Neuruppin-Alt Ruppin,96,52.9437,Brandenburg,2025-09-01
1,54.0,13.9908,1908-05-17,True,Angermünde,164,53.0316,Brandenburg,2025-09-01
2,55.0,13.4997,1993-09-06,True,Baruth,303,52.0613,Brandenburg,2025-09-01
3,46.0,13.5306,1957-01-01,True,Berlin,427,52.3807,Brandenburg,2025-09-01
4,69.0,14.3168,1887-01-01,True,Cottbus,880,51.7759,Brandenburg,2025-09-01
